# Notebook de submission (plantilla)

Estructura del notebook final reproducible. El modelo de la sección 5 es el del starter y está solo como **relleno**: se reemplaza por el modelo elegido por el equipo sin tocar el resto.

Flujo: configuración y semillas, rutas, datos, preprocesado, modelo, entrenamiento, validación, predicción de test, `submission.csv`, guardado de `models/modelo_final.pt` según el contrato modelo-app y comprobación de recarga.

Funciona en Kaggle, Colab y local:
- **Kaggle:** adjuntar los datos de la competencia; las salidas quedan en `/kaggle/working`.
- **Colab:** subir la carpeta `data/` a `/content/data` (o montar Drive y ajustar `DATA_DIR`); las salidas quedan en `/content`.
- **Local:** ejecutar desde `train/` o desde la raíz del repo; los datos se buscan en `data/` y las salidas van a la raíz del repo.

Modo rápido (prueba de punta a punta, 2 épocas): variable de entorno `QUICK=1`. En ese modo los archivos de salida llevan el sufijo `_quick` para no pisar el modelo final.

## 1. Configuración y semillas

In [ ]:
import glob, json, os, platform, random, sys, time
import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F

SEED = 42
QUICK = os.environ.get('QUICK') == '1'
EPOCHS = 2 if QUICK else 25
BATCH = 32
LR = 2e-3
CORTE_VAL = '2024061600'          # validación temporal: times >= corte
VARS = ['RH', 'T', 'U', 'V']
RMSE_CLIM_OFICIAL = np.array([25.360, 2.0652, 2.6776, 2.0876])

def fijar_semillas(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

fijar_semillas(SEED)
if os.environ.get('TORCH_THREADS'):
    torch.set_num_threads(int(os.environ['TORCH_THREADS']))
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'QUICK={QUICK} · épocas={EPOCHS} · device={dev} · torch {torch.__version__} · numpy {np.__version__} · python {platform.python_version()}')

## 2. Rutas (Kaggle, Colab o local)

In [ ]:
EN_KAGGLE = os.path.isdir('/kaggle/input')
EN_COLAB = (not EN_KAGGLE) and ('google.colab' in sys.modules or os.path.isdir('/content/sample_data'))

def raiz_repo():
    # Sube desde la carpeta actual hasta encontrar data/ o models/ (sirve desde train/ o desde la raíz)
    d = os.path.abspath(os.getcwd())
    for _ in range(4):
        if os.path.isdir(os.path.join(d, 'data')) or os.path.isdir(os.path.join(d, 'models')):
            return d
        d = os.path.dirname(d)
    return os.path.abspath(os.getcwd())

if EN_KAGGLE:
    DATA_DIRS, OUT_DIR = ['/kaggle/input'], '/kaggle/working'
elif EN_COLAB:
    DATA_DIRS, OUT_DIR = ['/content/data', '/content/drive/MyDrive', '/content'], '/content'
else:
    DATA_DIRS, OUT_DIR = [os.path.join(raiz_repo(), 'data')], raiz_repo()
if os.environ.get('DATA_DIR'):
    DATA_DIRS = [os.environ['DATA_DIR']] + DATA_DIRS

def find(name):
    for d in DATA_DIRS:
        hits = sorted(glob.glob(os.path.join(d, '**', name), recursive=True))
        if hits:
            return hits[0]
    raise FileNotFoundError(f'No encuentro {name} en {DATA_DIRS}')

MODELS_DIR = os.path.join(OUT_DIR, 'models'); os.makedirs(MODELS_DIR, exist_ok=True)
SUF = '_quick' if QUICK else ''
RUTA_SUB = os.path.join(OUT_DIR, f'submission{SUF}.csv')
RUTA_MODELO = os.path.join(MODELS_DIR, f'modelo_final{SUF}.pt')
RUTA_PRED_VAL = os.path.join(OUT_DIR, f'pred_val{SUF}.npz')
print('Entorno:', 'Kaggle' if EN_KAGGLE else 'Colab' if EN_COLAB else 'local')
print('Datos en:', DATA_DIRS, '\nSalidas:', RUTA_SUB, RUTA_MODELO, RUTA_PRED_VAL, sep='\n  ')

## 3. Datos

In [ ]:
tr = np.load(find('train.npz')); te = np.load(find('test.npz')); st = np.load(find('static.npz'))
Xtr, Ytr, ttr = tr['X'], tr['Y'], tr['times'].astype(str)
Xte, tte = te['X'], te['times'].astype(str)
dem = st['dem']
print('train X', Xtr.shape, 'Y', Ytr.shape, ttr[0], '->', ttr[-1])
print('test  X', Xte.shape, tte[0], '->', tte[-1])

## 4. Preprocesado (ventanas, split temporal, climatología, normalización)

In [ ]:
hour = lambda t: np.array([int(k[-2:]) for k in t])

def windows(X, t):
    # Ventanas t-2, t-1, t; si falta una hora previa se repite la hora actual (igual que el starter)
    ts = pd.to_datetime(t, format='%Y%m%d%H')
    pos = {k: i for i, k in enumerate(ts)}
    out = np.empty((len(X), 3) + X.shape[1:], np.float32)
    for i, k in enumerate(ts):
        for j, lag in enumerate([2, 1, 0]):
            idx = pos.get(k - pd.Timedelta(hours=lag), None)
            out[i, j] = X[idx] if idx is not None else X[i]
    return out

val = ttr >= CORTE_VAL; trn = ~val

def climatology(Y, t):
    h = hour(t)
    return np.stack([Y[h == k].mean(0) for k in range(24)])   # (24,70,35,4)

Wtr, Wte = windows(Xtr, ttr), windows(Xte, tte)

# Estadísticas SOLO con la parte de entrenamiento (se guardan en el checkpoint)
NORM = {
    'mu_x': Xtr[trn].mean((0, 1, 2)).astype(np.float32),                       # (4,)
    'sd_x': Xtr[trn].std((0, 1, 2)).astype(np.float32),                        # (4,)
    'clim': climatology(Ytr[trn], ttr[trn]).astype(np.float32),                # (24,70,35,4)
}
NORM['anom_sd'] = (Ytr[trn] - NORM['clim'][hour(ttr[trn])]).std((0, 1, 2)).astype(np.float32)  # (4,)
NORM['dem_mu'], NORM['dem_sd'] = float(dem.mean()), float(dem.std())

clim_rmse_val = np.sqrt(((NORM['clim'][hour(ttr[val])] - Ytr[val]) ** 2).mean(axis=(0, 1, 2)))

def local_score(pred, true):
    rmse = np.sqrt(((pred - true) ** 2).mean(axis=(0, 1, 2)))
    return 1 - (rmse / clim_rmse_val).mean(), 1 - (rmse / RMSE_CLIM_OFICIAL).mean(), dict(zip(VARS, rmse.round(3)))

print('ventanas', Wtr.shape, Wte.shape, '· train', trn.sum(), '· val', val.sum())
print('climatología en val:', local_score(NORM['clim'][hour(ttr[val])], Ytr[val]))

## 5. Modelo (RELLENO: CNN del starter)

Reemplazar esta celda por el modelo final. Requisitos para que el resto funcione:
- `construir_modelo(cfg, static)` devuelve un `nn.Module` cuyo `forward(x, h)` recibe `x` normalizado `(B,3,17,13,4)` y `h` hora UTC `(B,)` float, y devuelve la anomalía normalizada `(B,70,35,4)`.
- Todo lo fijo que el modelo necesita (grilla de interpolación, DEM) va como `buffer`, para que viaje dentro del `state_dict`.
- Si la salida no es anomalía normalizada, ajustar `desnormalizar()`.

In [ ]:
MODEL_CFG = {'arch': 'starter_cnn', 'w': 64}

def static_inputs(st, norm):
    glat, glon = st['gfs_lats'], st['gfs_lons']     # GFS fila 0 = norte
    wlat, wlon = st['wrf_lats'], st['wrf_lons']     # WRF fila 0 = sur
    LAT, LON = np.meshgrid(wlat, wlon, indexing='ij')
    gy = (LAT - glat[0]) / (glat[-1] - glat[0]) * 2 - 1
    gx = (LON - glon[0]) / (glon[-1] - glon[0]) * 2 - 1
    grid = np.stack([gx, gy], -1)[None].astype(np.float32)                       # (1,70,35,2)
    demn = ((st['dem'] - norm['dem_mu']) / norm['dem_sd'])[None, None].astype(np.float32)  # (1,1,70,35)
    return {'grid': grid, 'dem': demn}

class StarterNet(nn.Module):
    def __init__(self, grid, demn, w=64):
        super().__init__()
        self.register_buffer('grid', torch.as_tensor(grid))
        self.register_buffer('demn', torch.as_tensor(demn))
        self.enc = nn.Sequential(nn.Conv2d(14, w, 3, padding=1), nn.GELU(), nn.Conv2d(w, w, 3, padding=1), nn.GELU())
        self.static = nn.Parameter(torch.zeros(1, 8, 70, 35))
        self.dec = nn.Sequential(nn.Conv2d(w + 9, w, 3, padding=1), nn.GELU(),
                                 nn.Conv2d(w, w, 3, padding=1), nn.GELU(), nn.Conv2d(w, 4, 1))
    def forward(self, x, h):
        B = x.shape[0]
        x = x.permute(0, 1, 4, 2, 3).reshape(B, 12, 17, 13)
        ang = 2 * np.pi * h / 24
        hh = torch.stack([torch.sin(ang), torch.cos(ang)], 1)[..., None, None].expand(B, 2, 17, 13)
        f = self.enc(torch.cat([x, hh], 1))
        f = F.grid_sample(f, self.grid.expand(B, -1, -1, -1), align_corners=True)
        f = torch.cat([f, self.demn.expand(B, -1, -1, -1), self.static.expand(B, -1, -1, -1)], 1)
        return self.dec(f).permute(0, 2, 3, 1)

def construir_modelo(cfg, static):
    return StarterNet(static['grid'], static['dem'], w=cfg['w'])

STATIC = static_inputs(st, NORM)
model = construir_modelo(MODEL_CFG, STATIC).to(dev)
print(MODEL_CFG, f"{sum(p.numel() for p in model.parameters())/1e3:.0f} k parámetros")

## 6. Funciones de predicción (entrada y salida en unidades originales)

In [ ]:
def a_tensor(a): return torch.as_tensor(np.asarray(a), dtype=torch.float32, device=dev)

def desnormalizar(salida, h, clim_t, asd_t):
    return clim_t[h] + salida * asd_t

def predecir(model, norm, W, horas, bs=128):
    # W: (N,3,17,13,4) GFS crudo; horas: (N,) hora UTC entera -> (N,70,35,4) RH %, T K, U, V m/s
    mu, sd = a_tensor(norm['mu_x']), a_tensor(norm['sd_x'])
    clim_t, asd_t = a_tensor(norm['clim']), a_tensor(norm['anom_sd'])
    model.eval(); out = []
    with torch.no_grad():
        for s in range(0, len(W), bs):
            x = (a_tensor(W[s:s+bs]) - mu) / sd
            h = torch.as_tensor(np.asarray(horas[s:s+bs]), dtype=torch.long, device=dev)
            out.append(desnormalizar(model(x, h.float()), h, clim_t, asd_t).cpu().numpy())
    return np.concatenate(out)

## 7. Entrenamiento (selección por score en validación)

In [ ]:
fijar_semillas(SEED)
model = construir_modelo(MODEL_CFG, STATIC).to(dev)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCHS * int(np.ceil(trn.sum() / BATCH)))
mu_t, sd_t = a_tensor(NORM['mu_x']), a_tensor(NORM['sd_x'])
CLIM_T, ASD_T = a_tensor(NORM['clim']), a_tensor(NORM['anom_sd'])
rng = np.random.default_rng(SEED)
itr = np.where(trn)[0]
best, best_ep, best_state, historial = -9e9, -1, None, []
for ep in range(EPOCHS):
    model.train(); rng.shuffle(itr); t0 = time.time()
    for s in range(0, len(itr), BATCH):
        b = itr[s:s+BATCH]
        x = (a_tensor(Wtr[b]) - mu_t) / sd_t
        h = torch.as_tensor(hour(ttr[b]), dtype=torch.long, device=dev)
        y = (a_tensor(Ytr[b]) - CLIM_T[h]) / ASD_T
        loss = F.mse_loss(model(x, h.float()), y)
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    sc, sc_of, per_var = local_score(predecir(model, NORM, Wtr[val], hour(ttr[val])), Ytr[val])
    historial.append({'epoca': ep + 1, 'loss': float(loss.item()), 'score_val': float(sc), 'score_ref_oficial': float(sc_of)})
    if sc > best:
        best, best_ep, best_state = sc, ep + 1, {k: v.detach().clone() for k, v in model.state_dict().items()}
    print(f'época {ep+1:02d} · loss {loss.item():.3f} · score val {sc:.4f} (ref. oficial {sc_of:.4f}) · {per_var} · {time.time()-t0:.0f}s')
model.load_state_dict(best_state)
print('Mejor época', best_ep, '· score val', round(best, 4))

## 8. Validación: guardar predicciones para `tools/score_local.py`

In [ ]:
P_val = predecir(model, NORM, Wtr[val], hour(ttr[val]))
np.savez_compressed(RUTA_PRED_VAL, pred=P_val.astype(np.float32), times=ttr[val])
SCORE_VAL = local_score(P_val, Ytr[val])
print('score val local / ref. oficial / RMSE:', SCORE_VAL)
print('Guardado', RUTA_PRED_VAL)

## 9. Predicción de test y `submission.csv`

In [ ]:
sample = pd.read_csv(find('sample_submission.csv'), dtype={'id': str})
P = predecir(model, NORM, Wte, hour(tte))                       # (792,70,35,4)
pos = {k: i for i, k in enumerate(tte)}
parts = sample['id'].str.split('_', expand=True)
ti = parts[0].map(pos).values; r = parts[1].astype(int).values; c = parts[2].astype(int).values
assert not np.isnan(ti.astype(float)).any(), 'Hay horas del sample que no están en test'
sub = pd.DataFrame({'id': sample['id']})
for k, v in enumerate(VARS):
    sub[v] = P[ti.astype(int), r, c, k].round(3)

# Comprobaciones mínimas (las completas están en tools/verificar_submission.py)
assert len(sub) == 646_800 and sub['id'].is_unique and (sub['id'].values == sample['id'].values).all()
assert np.isfinite(sub[VARS].values).all()
n_rh = int((~sub['RH'].between(0, 100)).sum())
if n_rh:
    print(f'AVISO: {n_rh} valores de RH fuera de [0,100]; se recortan')
sub['RH'] = sub['RH'].clip(0, 100)
assert sub['T'].mean() > 150, 'T debe ir en Kelvin'
sub.to_csv(RUTA_SUB, index=False)
print(sub.shape, '->', RUTA_SUB); sub.describe().loc[['min', 'mean', 'max']]

## 10. Guardar `models/modelo_final.pt` (contrato modelo-app)

Contenido del checkpoint:
- `state_dict`: pesos (incluye como buffers la grilla de interpolación y el DEM normalizado).
- `model_cfg`: arquitectura y parámetros para reconstruir el modelo.
- `norm`: `mu_x`, `sd_x` (4,), `clim` (24,70,35,4), `anom_sd` (4,), `dem_mu`, `dem_sd`.
- `contrato`: entrada `(N,3,17,13,4)` GFS crudo con ventanas t-2, t-1, t + hora UTC `(N,)` → salida `(N,70,35,4)` en unidades originales; orden de variables y orientación de filas.
- `meta`: semilla, épocas, mejor época, score de validación, versiones.

Sin estado del optimizador.

In [ ]:
ckpt = {
    'state_dict': {k: v.detach().cpu() for k, v in model.state_dict().items()},
    'model_cfg': MODEL_CFG,
    'norm': {k: (torch.as_tensor(v) if isinstance(v, np.ndarray) else v) for k, v in NORM.items()},
    'contrato': {
        'entrada': 'x (N,3,17,13,4) float32 GFS sin normalizar, ventanas [t-2, t-1, t]; horas (N,) int hora UTC 0-23',
        'salida': '(N,70,35,4) float32 en unidades originales',
        'vars': VARS, 'unidades': ['%', 'K', 'm/s', 'm/s'],
        'orientacion': 'GFS fila 0 = norte; salida WRF fila 0 = sur; columna 0 = oeste',
        'salida_modelo': 'anomalia normalizada: y = clim[hora] + salida * anom_sd',
    },
    'meta': {
        'seed': SEED, 'epochs': EPOCHS, 'best_epoch': best_ep, 'quick': QUICK,
        'score_val_local': float(SCORE_VAL[0]), 'score_val_ref_oficial': float(SCORE_VAL[1]),
        'corte_val': CORTE_VAL, 'historial': historial,
        'torch': torch.__version__, 'numpy': np.__version__, 'python': platform.python_version(),
    },
}
torch.save(ckpt, RUTA_MODELO)
print('Guardado', RUTA_MODELO, f'({os.path.getsize(RUTA_MODELO)/1e6:.2f} MB)')

## 11. Comprobación: recargar el checkpoint como lo haría la app

In [ ]:
ck = torch.load(RUTA_MODELO, map_location=dev, weights_only=False)
norm2 = {k: (v.numpy() if torch.is_tensor(v) else v) for k, v in ck['norm'].items()}
static2 = {'grid': ck['state_dict']['grid'].numpy(), 'dem': ck['state_dict']['demn'].numpy()}
m2 = construir_modelo(ck['model_cfg'], static2).to(dev)
m2.load_state_dict(ck['state_dict'])
P2 = predecir(m2, norm2, Wte[:16], hour(tte[:16]))
dif = np.abs(P2 - P[:16]).max()
print('Diferencia máxima con la predicción original:', dif)
assert dif < 1e-4, 'El checkpoint recargado no reproduce la predicción'
print('OK: checkpoint autocontenido')